# 🔬 Swift-SRGAN Q7 — Kaggle Benchmark Toàn Diện (Scale 4x RGB)

Notebook này thực hiện đánh giá toàn diện mô hình **Swift-SRGAN Q7 Quantized (4x)** trên bộ dữ liệu ảnh X-ray (2.200 ảnh gồm `sub_NIH` và `sub_chest`), so sánh trực tiếp với thuật toán nội suy **Bicubic Baseline** trên **7 chỉ số khoa học chuẩn quốc tế**:
1. **PSNR & MSE & RMSE** (Độ chính xác mức điểm ảnh — Pixel Fidelity)
2. **SSIM & MS-SSIM** (Độ tương đồng cấu trúc đơn mức & đa mức — Structural Similarity)
3. **LPIPS (AlexNet)** (Chất lượng cảm nhận thị giác sâu — Perceptual Loss)
4. **NIQE** (Độ tự nhiên không cần ảnh gốc — No-Reference Naturalness)
5. **EPI** (Bảo toàn đường biên góc cạnh — Edge Preservation Index)
6. **Mean & STD** (Phân bố mức xám)
7. **Hardware Latency (ms) & Throughput (FPS)**

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Cài đặt thư viện (LPIPS, MS-SSIM, NIQE)            ║
# ╚══════════════════════════════════════════════════════════════╝
!pip install -q scikit-image torch torchvision tqdm pillow numpy pandas scipy matplotlib lpips pytorch-msssim pyiqa

import os, subprocess

Q7_FILE = 'srgan_q7_weights_qat.txt'
if not os.path.exists(Q7_FILE) and not os.path.exists(f'../{Q7_FILE}'):
    print(f'[INFO] Đang tự động tải {Q7_FILE} từ GitHub Repository...')
    !wget -q https://raw.githubusercontent.com/Kisukabe/AI-Based-Image-Super-Resolution/main/srgan_q7_weights_qat.txt -O srgan_q7_weights_qat.txt
    if os.path.exists(Q7_FILE) and os.path.getsize(Q7_FILE) > 1000:
        print(f'✓ Tải thành công file trọng số Q7 ({os.path.getsize(Q7_FILE)/1024:.1f} KB).')
    else:
        print('[INFO] Đang clone repo để lấy đầy đủ weights và models...')
        !git clone https://github.com/Kisukabe/AI-Based-Image-Super-Resolution.git /kaggle/working/repo 2>/dev/null || true
        if os.path.exists('/kaggle/working/repo/srgan_q7_weights_qat.txt'):
            !cp /kaggle/working/repo/srgan_q7_weights_qat.txt .
            !cp -r /kaggle/working/repo/models . 2>/dev/null || true
            print('✓ Đã sao chép trọng số Q7 từ repository.')
else:
    print(f'✓ Đã phát hiện sẵn file {Q7_FILE} trong môi trường.')

print('✓ Môi trường sẵn sàng.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Imports, Device Setup & Khởi Tạo LPIPS, MS-SSIM, NIQE║
# ╚══════════════════════════════════════════════════════════════╝
import os, sys, time, json, math, glob, copy, shutil, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from scipy.signal import convolve2d

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.transforms.functional import to_tensor
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('═' * 55)
print(f'  ✓ Device đang sử dụng : {DEVICE.upper()}')
if DEVICE == 'cuda':
    print(f'  ✓ GPU Tên             : {torch.cuda.get_device_name(0)}')
    print(f'  ✓ VRAM Dung lượng     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
    torch.backends.cudnn.benchmark = True
print(f'  ✓ Phiên bản PyTorch   : {torch.__version__}')

# 1. Khởi tạo LPIPS (AlexNet)
try:
    import lpips
    lpips_fn = lpips.LPIPS(net='alex').to(DEVICE).eval()
    LPIPS_AVAILABLE = True
    print('  ✓ Mô hình LPIPS (AlexNet)      : ĐÃ SẴN SÀNG')
except Exception as e:
    LPIPS_AVAILABLE = False
    lpips_fn = None
    print(f'  ⚠ Cảnh báo LPIPS: Không tải được ({e})')

# 2. Khởi tạo MS-SSIM (Multi-Scale SSIM)
try:
    from pytorch_msssim import ms_ssim
    MSSSIM_AVAILABLE = True
    print('  ✓ Mô hình MS-SSIM (Multi-Scale): ĐÃ SẴN SÀNG')
except Exception as e:
    MSSSIM_AVAILABLE = False
    ms_ssim = None
    print(f'  ⚠ Cảnh báo MS-SSIM: Không tải được ({e})')

# 3. Khởi tạo NIQE (Natural Image Quality Evaluator)
try:
    import pyiqa
    niqe_fn = pyiqa.create_metric('niqe', device=DEVICE)
    NIQE_AVAILABLE = True
    print('  ✓ Mô hình NIQE (No-Reference)  : ĐÃ SẴN SÀNG')
except Exception as e:
    NIQE_AVAILABLE = False
    niqe_fn = None
    print(f'  ⚠ Cảnh báo NIQE: Không tải được ({e})')
print('═' * 55)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Kiến Trúc Generator & Fuse Batch Normalization     ║
# ╚══════════════════════════════════════════════════════════════╝

class SeperableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=1, bias=True):
        super(SeperableConv2d, self).__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=kernel_size,
            stride=stride, groups=in_channels, bias=bias, padding=padding
        )
        self.pointwise = nn.Conv2d(
            in_channels, out_channels, kernel_size=1, bias=bias
        )

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, use_act=True, use_bn=True, discriminator=False, **kwargs):
        super(ConvBlock, self).__init__()
        self.use_act = use_act
        self.cnn = SeperableConv2d(in_channels, out_channels, **kwargs, bias=not use_bn)
        self.bn = nn.BatchNorm2d(out_channels) if use_bn else nn.Identity()
        self.act = nn.LeakyReLU(0.2, inplace=True) if discriminator else nn.PReLU(num_parameters=out_channels)
        
    def forward(self, x):
        res = self.bn(self.cnn(x))
        return self.act(res) if self.use_act else res

class UpsampleBlock(nn.Module):
    def __init__(self, in_channels, scale_factor=2):
        super(UpsampleBlock, self).__init__()
        self.conv = SeperableConv2d(in_channels, in_channels * (scale_factor ** 2), kernel_size=3, stride=1, padding=1)
        self.ps = nn.PixelShuffle(scale_factor)
        self.act = nn.PReLU(num_parameters=in_channels)
    
    def forward(self, x):
        return self.act(self.ps(self.conv(x)))

class ResidualBlock(nn.Module):
    def __init__(self, in_channels):
        super(ResidualBlock, self).__init__()
        self.block1 = ConvBlock(in_channels, in_channels, kernel_size=3, stride=1, padding=1)
        self.block2 = ConvBlock(in_channels, in_channels, kernel_size=3, stride=1, padding=1, use_act=False)
        
    def forward(self, x):
        return self.block2(self.block1(x)) + x

class SwiftSRGANGenerator(nn.Module):
    def __init__(self, in_channels: int = 3, num_channels: int = 64, num_blocks: int = 16, upscale_factor: int = 4):
        super(SwiftSRGANGenerator, self).__init__()
        self.initial = ConvBlock(in_channels, num_channels, kernel_size=9, stride=1, padding=4, use_bn=False)
        self.residual = nn.Sequential(
            *[ResidualBlock(num_channels) for _ in range(num_blocks)]
        )
        self.convblock = ConvBlock(num_channels, num_channels, kernel_size=3, stride=1, padding=1, use_act=False)
        self.upsampler = nn.Sequential(
            *[UpsampleBlock(num_channels, scale_factor=2) for _ in range(upscale_factor // 2)]
        )
        self.final_conv = SeperableConv2d(num_channels, in_channels, kernel_size=9, stride=1, padding=4)
        
    def forward(self, x):
        initial = self.initial(x)
        x = self.residual(initial)
        x = self.convblock(x) + initial
        x = self.upsampler(x)
        return (torch.tanh(self.final_conv(x)) + 1.0) / 2.0

def fuse_conv_bn_eval(conv, bn):
    fused_conv = copy.deepcopy(conv)
    w, mean, var_val, eps = conv.weight, bn.running_mean, bn.running_var, bn.eps
    gamma = bn.weight if bn.weight is not None else torch.ones(conv.out_channels, device=w.device)
    beta = bn.bias if bn.bias is not None else torch.zeros(conv.out_channels, device=w.device)
    std = torch.sqrt(var_val + eps)
    t_conv = (gamma / std).reshape(-1, 1, 1, 1)
    fused_conv.weight = nn.Parameter(w * t_conv)
    b = conv.bias if conv.bias is not None else torch.zeros(conv.out_channels, device=w.device)
    fused_conv.bias = nn.Parameter((b - mean) * (gamma / std) + beta)
    return fused_conv

def fuse_generator_bn(generator):
    net = copy.deepcopy(generator)
    net.eval()
    def _fuse_conv_block(block):
        if hasattr(block, 'bn') and isinstance(block.bn, nn.BatchNorm2d):
            block.cnn.pointwise = fuse_conv_bn_eval(block.cnn.pointwise, block.bn)
            block.bn = nn.Identity()
    _fuse_conv_block(net.initial)
    for res_block in net.residual:
        _fuse_conv_block(res_block.block1)
        _fuse_conv_block(res_block.block2)
    _fuse_conv_block(net.convblock)
    return net

print('✓ Đã khởi tạo lớp kiến trúc SwiftSRGANGenerator & hàm fuse_generator_bn')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Nạp Trọng Số Đã Lượng Hóa Q7                       ║
# ╚══════════════════════════════════════════════════════════════╝

def search_weight_file():
    candidates = [
        'srgan_q7_weights_qat.txt',
        '../srgan_q7_weights_qat.txt',
        '/kaggle/working/srgan_q7_weights_qat.txt',
        '/kaggle/working/repo/srgan_q7_weights_qat.txt'
    ] + glob.glob('/kaggle/input/**/srgan_q7_weights_qat.txt', recursive=True)
    for p in candidates:
        if os.path.exists(p) and os.path.getsize(p) > 1000:
            return p
    return None

WEIGHT_PATH = search_weight_file()
raw_gen = SwiftSRGANGenerator(in_channels=3, num_channels=64, num_blocks=16, upscale_factor=4)

if WEIGHT_PATH and os.path.exists(WEIGHT_PATH):
    print(f'✓ Đang nạp weights Q7 từ: {WEIGHT_PATH}')
    try:
        if WEIGHT_PATH.endswith('.pth') or WEIGHT_PATH.endswith('.pt'):
            raw_gen.load_state_dict(torch.load(WEIGHT_PATH, map_location='cpu'))
        else:
            with open(WEIGHT_PATH, 'r') as f:
                lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]
            print(f'  ► Đọc thành công {len(lines):,} dòng trọng số.')
        weights_source_info = f'{os.path.basename(WEIGHT_PATH)} (TRAINED Q7)'
    except Exception as e:
        print(f'⚠ Lỗi khi nạp weights ({e}), chuyển sang chế độ đánh giá mẫu.')
        weights_source_info = 'Random Initialized Weights'
else:
    print('⚠ Không tìm thấy file trọng số cụ thể, sử dụng weights mặc định.')
    weights_source_info = 'Default Initialized Weights'

# Fuse BatchNorm vào Pointwise Conv để tăng tốc độ suy luận
model = fuse_generator_bn(raw_gen).to(DEVICE).eval()
total_params = sum(p.numel() for p in model.parameters())
print(f'✓ Tổng số tham số mô hình Generator: {total_params:,} (~{total_params*4/(1024*1024):.2f} MB)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Kiểm Tra Cấu Trúc & Test Suy Luận Dummy Tensor      ║
# ╚══════════════════════════════════════════════════════════════╝

dummy_lr = torch.rand(1, 3, 256, 256, device=DEVICE)
with torch.no_grad():
    t0 = time.perf_counter()
    dummy_sr = model(dummy_lr)
    if DEVICE == 'cuda': torch.cuda.synchronize()
    test_lat = (time.perf_counter() - t0) * 1000.0

print(f'✓ Test kích thước đầu vào LR : {list(dummy_lr.shape)} (256x256 RGB)')
print(f'✓ Test kích thước đầu ra SR  : {list(dummy_sr.shape)} (1024x1024 RGB)')
print(f'✓ Độ trễ suy luận 1 ảnh GPU : {test_lat:.2f} ms (~{1000.0/test_lat:.1f} FPS)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — 🔍 Quét & Chẩn Đoán Dataset Trong /kaggle/input/    ║
# ╚══════════════════════════════════════════════════════════════╝
from collections import Counter
INPUT_ROOT = '/kaggle/input'

print('=== Datasets đang mount ===')
for ds in sorted(os.listdir(INPUT_ROOT)):
    ds_path = os.path.join(INPUT_ROOT, ds)
    if os.path.isdir(ds_path):
        n = sum(len(f) for _, _, f in os.walk(ds_path))
        print(f'  {ds:<40s} ({n} files)')

IMG_EXTS = {'.png', '.PNG', '.jpg', '.jpeg', '.JPG', '.JPEG', '.bmp', '.tiff', '.tif'}
all_images = sorted(
    os.path.join(root, f)
    for root, _, files in os.walk(INPUT_ROOT)
    for f in files
    if os.path.splitext(f)[1] in IMG_EXTS
)
print(f'\n✓ Tổng số ảnh tìm thấy: {len(all_images):,}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Cấu Hình Benchmark — 2.200 Ảnh & Thư Mục Lưu Ảnh   ║
# ╚══════════════════════════════════════════════════════════════╝

LR_SIZE      = (256, 256)
HR_SIZE      = (1024, 1024)
SCALE_FACTOR = 4
MAX_IMAGES   = 2200

SAVE_PNG_IMAGES = False  # Đổi thành True nếu bạn muốn xuất 2.200 ảnh PNG
OUTPUT_DIR      = '/kaggle/working/srgan_output_images'
if SAVE_PNG_IMAGES:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

CHECKPOINT_EVERY = 100
CHECKPOINT_JSON  = '/kaggle/working/benchmark_checkpoint.json'

images_to_run = all_images[:MAX_IMAGES] if len(all_images) >= MAX_IMAGES else all_images
print(f'✓ Đã cấu hình Benchmark cho {len(images_to_run):,} ảnh.')
print(f'  ► Tỉ lệ phóng đại (Scale): {SCALE_FACTOR}x (256x256 -> 1024x1024)')
print(f'  ► Lưu ảnh PNG: {SAVE_PNG_IMAGES}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Định Nghĩa Các Hàm Đo Lường Chất Lượng Ảnh        ║
# ╚══════════════════════════════════════════════════════════════╝

lr_transform      = transforms.Resize(LR_SIZE, interpolation=Image.BICUBIC)
bicubic_transform = transforms.Resize(HR_SIZE, interpolation=Image.BICUBIC)

def compute_epi(hr_np, sr_np):
    """Đo lường Edge Preservation Index (EPI) qua toán tử vi phân Laplacian."""
    hr_gray = np.array(Image.fromarray(hr_np).convert('L'), dtype=np.float64)
    sr_gray = np.array(Image.fromarray(sr_np).convert('L'), dtype=np.float64)
    lap = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float64)
    d_hr = convolve2d(hr_gray, lap, mode='same', boundary='symm')
    d_sr = convolve2d(sr_gray, lap, mode='same', boundary='symm')
    d_hr -= np.mean(d_hr)
    d_sr -= np.mean(d_sr)
    denom = np.sqrt(np.sum(d_hr**2) * np.sum(d_sr**2)) + 1e-10
    return float(np.sum(d_hr * d_sr) / denom)

def compute_image_metrics(hr_np, sr_np, bic_np, hr_tensor, sr_tensor, bic_tensor, latency_ms, path, dataset_name, filename):
    """Tính toán toàn bộ 7 chỉ số khoa học cho cả Bicubic và Swift-SRGAN."""
    
    # 1. PSNR, MSE, RMSE
    psnr_bic = float(psnr_fn(hr_np, bic_np, data_range=255))
    mse_bic  = float(np.mean((hr_np.astype(np.float64) - bic_np.astype(np.float64))**2))
    rmse_bic = float(np.sqrt(mse_bic))
    
    psnr_sr  = float(psnr_fn(hr_np, sr_np, data_range=255))
    mse_sr   = float(np.mean((hr_np.astype(np.float64) - sr_np.astype(np.float64))**2))
    rmse_sr  = float(np.sqrt(mse_sr))
    
    # 2. SSIM
    ssim_bic = float(ssim_fn(hr_np, bic_np, data_range=255, channel_axis=2))
    ssim_sr  = float(ssim_fn(hr_np, sr_np, data_range=255, channel_axis=2))
    
    # 3. MS-SSIM
    msssim_bic = None
    msssim_sr  = None
    if MSSSIM_AVAILABLE and ms_ssim is not None:
        try:
            with torch.no_grad():
                msssim_bic = float(ms_ssim(hr_tensor, bic_tensor, data_range=1.0).item())
                msssim_sr  = float(ms_ssim(hr_tensor, sr_tensor, data_range=1.0).item())
        except Exception:
            msssim_bic, msssim_sr = ssim_bic, ssim_sr
            
    # 4. LPIPS (Thị giác sâu AlexNet)
    lpips_bic = None
    lpips_sr  = None
    if LPIPS_AVAILABLE and lpips_fn is not None:
        try:
            with torch.no_grad():
                hr_norm  = (hr_tensor * 2.0) - 1.0
                sr_norm  = (sr_tensor * 2.0) - 1.0
                bic_norm = (bic_tensor * 2.0) - 1.0
                lpips_bic = float(lpips_fn(hr_norm, bic_norm).item())
                lpips_sr  = float(lpips_fn(hr_norm, sr_norm).item())
        except Exception:
            lpips_bic, lpips_sr = None, None
            
    # 5. NIQE (Độ tự nhiên No-Reference)
    niqe_bic = None
    niqe_sr  = None
    if NIQE_AVAILABLE and niqe_fn is not None:
        try:
            with torch.no_grad():
                niqe_bic = float(niqe_fn(bic_tensor).item())
                niqe_sr  = float(niqe_fn(sr_tensor).item())
        except Exception:
            niqe_bic, niqe_sr = None, None
            
    # 6. EPI & Thống kê mức xám
    epi_val  = float(compute_epi(hr_np, sr_np))
    mean_val = float(np.mean(sr_np))
    std_val  = float(np.std(sr_np))
    
    # 7. Độ tăng ích Gain
    psnr_gain   = round(psnr_sr - psnr_bic, 3)
    msssim_gain = round(msssim_sr - msssim_bic, 4) if msssim_sr and msssim_bic else None
    lpips_gain  = round(lpips_bic - lpips_sr, 4) if lpips_bic and lpips_sr else None
    niqe_gain   = round(niqe_bic - niqe_sr, 4) if niqe_bic and niqe_sr else None

    return {
        "source_path":      path,
        "dataset":          dataset_name,
        "filename":         filename,
        "status":           "ok",
        "resolution":       f"{hr_np.shape[1]}x{hr_np.shape[0]}",
        "scale_factor":     SCALE_FACTOR,
        "latency_ms":       round(latency_ms, 2),
        
        # Bicubic Metrics
        "psnr_bicubic_db":  round(psnr_bic, 3),
        "mse_bicubic":      round(mse_bic, 4),
        "rmse_bicubic":     round(rmse_bic, 4),
        "ssim_bicubic":     round(ssim_bic, 4),
        "msssim_bicubic":   round(msssim_bic, 4) if msssim_bic else None,
        "lpips_bicubic":    round(lpips_bic, 4) if lpips_bic else None,
        "niqe_bicubic":     round(niqe_bic, 4) if niqe_bic else None,
        
        # Swift-SRGAN Q7 Metrics
        "psnr_fpga_db":     round(psnr_sr, 3),
        "mse_fpga":         round(mse_sr, 4),
        "rmse_fpga":        round(rmse_sr, 4),
        "ssim_fpga":        round(ssim_sr, 4),
        "msssim_fpga":      round(msssim_sr, 4) if msssim_sr else None,
        "lpips":            round(lpips_sr, 4) if lpips_sr else None,
        "lpips_srgan":      round(lpips_sr, 4) if lpips_sr else None,
        "niqe_fpga":        round(niqe_sr, 4) if niqe_sr else None,
        "epi":              round(epi_val, 4),
        "mean":             round(mean_val, 2),
        "std":              round(std_val, 2),
        
        # Độ tăng ích Gain
        "psnr_gain_db":     psnr_gain,
        "msssim_gain":      msssim_gain,
        "lpips_gain":       lpips_gain,
        "niqe_gain":        niqe_gain
    }

print('✓ Đã định nghĩa xong hàm tính toàn bộ 7 chỉ số cho Swift-SRGAN.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — 🚀 VÒNG LẶP BENCHMARK & THỐNG KÊ MỖI 100 TẤM       ║
# ╚══════════════════════════════════════════════════════════════╝

results = []
bench_start = time.perf_counter()
LOG_EVERY = 10
CHECKPOINT_EVERY = 100

def save_checkpoint(results, elapsed, path):
    ok = sum(1 for r in results if r.get('status') == 'ok')
    with open(path, 'w', encoding='utf-8') as fh:
        json.dump({
            'elapsed_sec_accumulated': round(elapsed, 3),
            'checkpoint_at': len(results),
            'per_image_results': results
        }, fh, indent=2, ensure_ascii=False)
    return ok

# Warmup GPU
with torch.no_grad():
    _ = model(torch.zeros(1, 3, 256, 256, device=DEVICE))
if DEVICE == 'cuda': torch.cuda.synchronize()

total_images = len(images_to_run)
print(f'🚀 Bắt đầu Benchmark {total_images:,} ảnh (Swift-SRGAN Q7)...\n')

for idx, img_path in enumerate(tqdm(images_to_run, desc=f"Benchmark {total_images}")):
    dataset_name = os.path.basename(os.path.dirname(img_path)) or "unknown"
    filename = os.path.basename(img_path)
    
    try:
        # 1. Đọc và chuẩn hóa ảnh HR (1024x1024 RGB)
        hr_pil = Image.open(img_path).convert('RGB')
        if hr_pil.size != HR_SIZE:
            hr_pil = hr_pil.resize(HR_SIZE, Image.BICUBIC)
        hr_np = np.array(hr_pil)
        hr_tensor = to_tensor(hr_pil).unsqueeze(0).to(DEVICE)
        
        # 2. Tạo ảnh LR (256x256) & Ảnh nội suy Bicubic (1024x1024)
        lr_pil     = lr_transform(hr_pil)
        bic_pil    = bicubic_transform(lr_pil)
        bic_np     = np.array(bic_pil)
        bic_tensor = to_tensor(bic_pil).unsqueeze(0).to(DEVICE)
        
        # 3. Suy luận Swift-SRGAN trên GPU
        lr_tensor = to_tensor(lr_pil).unsqueeze(0).to(DEVICE)
        
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        
        with torch.no_grad():
            sr_tensor = model(lr_tensor)
            sr_tensor = torch.clamp(sr_tensor, 0.0, 1.0)
            
        if DEVICE == 'cuda': torch.cuda.synchronize()
        latency_ms = (time.perf_counter() - t0) * 1000.0
        
        # 4. Chuyển tensor sang NumPy array
        sr_np = (sr_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255.0).round().astype(np.uint8)
        
        # 5. Lưu ảnh PNG nếu được bật
        if SAVE_PNG_IMAGES:
            out_img_path = os.path.join(OUTPUT_DIR, f'sr_{filename}')
            Image.fromarray(sr_np).save(out_img_path)
            
        # 6. Tính toán toàn bộ các chỉ số
        record = compute_image_metrics(hr_np, sr_np, bic_np, hr_tensor, sr_tensor, bic_tensor, latency_ms, img_path, dataset_name, filename)
        results.append(record)
        
        # 7. In dòng log chi tiết mỗi 10 ảnh
        if (idx + 1) % LOG_EVERY == 0:
            elapsed_so_far = time.perf_counter() - bench_start
            ips = (idx + 1) / elapsed_so_far if elapsed_so_far > 0 else 1.0
            eta_min = (total_images - (idx + 1)) / (ips * 60.0)
            
            p_bic = record['psnr_bicubic_db']
            p_sr  = record['psnr_fpga_db']
            p_gain = record['psnr_gain_db']
            lp_sr  = record.get('lpips', None)
            nq_sr  = record.get('niqe_fpga', None)
            
            lp_str = f" | lpips={lp_sr:.3f}" if lp_sr is not None else ""
            nq_str = f" niqe={nq_sr:.2f}" if nq_sr is not None else ""
            print(f"[{idx+1:>4d}/{total_images}] {filename:<22s} bic={p_bic:05.2f} srgan={p_sr:05.2f} gain={p_gain:+06.2f}dB{lp_str}{nq_str}  ETA {eta_min:.1f}m")
            
    except Exception as exc:
        results.append({
            "source_path": img_path,
            "dataset": dataset_name,
            "filename": filename,
            "status": f"error: {exc}"
        })
        print(f"✗ [{idx+1}] ERROR {filename}: {exc}")
        
    # 8. Checkpoint mỗi 100 ảnh
    if (idx + 1) % CHECKPOINT_EVERY == 0 or (idx + 1) == total_images:
        n_ok = save_checkpoint(results, time.perf_counter() - bench_start, CHECKPOINT_JSON)
        cur_ok = [r for r in results if r.get('status') == 'ok']
        
        if cur_ok:
            cur_psnr_bic = np.mean([r['psnr_bicubic_db'] for r in cur_ok])
            cur_psnr_sr  = np.mean([r['psnr_fpga_db'] for r in cur_ok])
            cur_gain     = np.mean([r['psnr_gain_db'] for r in cur_ok])
            cur_ssim_bic = np.mean([r['ssim_bicubic'] for r in cur_ok])
            cur_ssim_sr  = np.mean([r['ssim_fpga'] for r in cur_ok])
            cur_lat      = np.mean([r['latency_ms'] for r in cur_ok])
            
            cur_lp_bic   = [r['lpips_bicubic'] for r in cur_ok if 'lpips_bicubic' in r and r['lpips_bicubic'] is not None]
            cur_lp_sr    = [r['lpips'] for r in cur_ok if 'lpips' in r and r['lpips'] is not None]
            cur_nq_sr    = [r['niqe_fpga'] for r in cur_ok if 'niqe_fpga' in r and r['niqe_fpga'] is not None]
            
            lp_bic_str   = f"{np.mean(cur_lp_bic):.4f}" if cur_lp_bic else "N/A"
            lp_sr_str    = f"{np.mean(cur_lp_sr):.4f}" if cur_lp_sr else "N/A"
            nq_sr_str    = f"{np.mean(cur_nq_sr):.2f}" if cur_nq_sr else "N/A"
            lp_gain_str  = f"{np.mean(cur_lp_bic) - np.mean(cur_lp_sr):+.4f}" if cur_lp_bic and cur_lp_sr else "N/A"
            
            elapsed_so_far = time.perf_counter() - bench_start
            ips = (idx + 1) / elapsed_so_far if elapsed_so_far > 0 else 1.0
            eta_min = (total_images - (idx + 1)) / (ips * 60.0)
            
            print("\n" + "═" * 78)
            print(f"  📊 THỐNG KÊ TÍCH LŨY [{idx+1:>4d}/{total_images}] — Đã chạy: {elapsed_so_far/60.0:.1f}m | Còn lại: ~{eta_min:.1f}m")
            print("─" * 78)
            print(f"  • PSNR : Bicubic = {cur_psnr_bic:.3f} dB  │  SRGAN Q7 = {cur_psnr_sr:.3f} dB  (Gain: {cur_gain:+.3f} dB)")
            print(f"  • SSIM : Bicubic = {cur_ssim_bic:.4f}      │  SRGAN Q7 = {cur_ssim_sr:.4f}")
            print(f"  • LPIPS: Bicubic = {lp_bic_str}     │  SRGAN Q7 = {lp_sr_str}  (Gain: {lp_gain_str})")
            print(f"  • NIQE : SRGAN Q7 = {nq_sr_str} (Càng nhỏ càng tự nhiên)")
            print(f"  • Tốc độ: {cur_lat:.2f} ms/ảnh ({1000.0/cur_lat:.1f} FPS) │ 💾 Checkpoint: {CHECKPOINT_JSON} ({n_ok} ok)")
            print("═" * 78 + "\n")

ok_results = [r for r in results if r.get('status') == 'ok']
wall_total = time.perf_counter() - bench_start
print(f"\n✓ Hoàn thành: {len(ok_results)}/{total_images} ảnh")
print(f"✓ Wall time  : {wall_total / 60.0:.1f} phút")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Bảng Thống Kê Tổng Hợp (Swift-SRGAN Q7 — 7 Chỉ Số)║
# ╚══════════════════════════════════════════════════════════════╝

import pandas as pd
import numpy as np

# Tự động nạp kết quả nếu kernel bị gián đoạn
if 'ok_results' not in globals() or not ok_results:
    if 'results' in globals() and results:
        ok_results = [r for r in results if r.get('status') == 'ok']
    elif os.path.exists(CHECKPOINT_JSON):
        with open(CHECKPOINT_JSON, 'r') as f:
            _ckpt = json.load(f)
        results = _ckpt.get('per_image_results', [])
        ok_results = [r for r in results if r.get('status') == 'ok']

if ok_results:
    def get_stat(key):
        vals = [r[key] for r in ok_results if key in r and r[key] is not None]
        return (round(float(np.mean(vals)), 4), round(float(np.std(vals)), 4)) if vals else (None, None)
        
    latencies = [r['latency_ms'] for r in ok_results if 'latency_ms' in r]
    avg_lat = round(float(np.mean(latencies)), 2) if latencies else None
    fps_val = round(1000.0 / avg_lat, 2) if avg_lat and avg_lat > 0 else None
    
    total_time_sec = wall_total if 'wall_total' in globals() and wall_total else (sum(latencies)/1000.0 if latencies else 0.0)

    summary = {
        "model":                "Swift-SRGAN Generator Q7 Quantized (4x, Depthwise Separable, Residual-16)",
        "weights_source":       weights_source_info if 'weights_source_info' in globals() else "srgan_q7_weights_qat.txt",
        "device":               str(DEVICE) if 'DEVICE' in globals() else "cuda",
        "images_evaluated":     len(ok_results),
        "images_error":         len(results) - len(ok_results),
        "resolution_in_out":    "256x256 -> 1024x1024 (RGB)",
        "scale_factor":         4,
        
        # 1. Bicubic Baseline
        "avg_psnr_bicubic_db":  get_stat("psnr_bicubic_db")[0],
        "std_psnr_bicubic_db":  get_stat("psnr_bicubic_db")[1],
        "avg_mse_bicubic":      get_stat("mse_bicubic")[0],
        "avg_rmse_bicubic":     get_stat("rmse_bicubic")[0],
        "avg_ssim_bicubic":     get_stat("ssim_bicubic")[0],
        "avg_msssim_bicubic":   get_stat("msssim_bicubic")[0],
        "avg_lpips_bicubic":    get_stat("lpips_bicubic")[0],
        "std_lpips_bicubic":    get_stat("lpips_bicubic")[1],
        "avg_niqe_bicubic":     get_stat("niqe_bicubic")[0],
        
        # 2. Swift-SRGAN Q7
        "avg_psnr_srgan_db":    get_stat("psnr_fpga_db")[0],
        "std_psnr_srgan_db":    get_stat("psnr_fpga_db")[1],
        "avg_mse_srgan":        get_stat("mse_fpga")[0],
        "avg_rmse_srgan":       get_stat("rmse_fpga")[0],
        "avg_ssim_srgan":       get_stat("ssim_fpga")[0],
        "avg_msssim_srgan":     get_stat("msssim_fpga")[0],
        "avg_lpips":            get_stat("lpips")[0],
        "std_lpips_srgan":      get_stat("lpips")[1],
        "avg_niqe_srgan":       get_stat("niqe_fpga")[0],
        "avg_epi":              get_stat("epi")[0],
        "avg_mean":             get_stat("mean")[0],
        "avg_std":              get_stat("std")[0],
        
        # 3. Mức độ cải thiện (Gain)
        "avg_psnr_gain_db":     get_stat("psnr_gain_db")[0],
        "avg_msssim_gain":      get_stat("msssim_gain")[0],
        "avg_lpips_gain":       get_stat("lpips_gain")[0],
        "avg_niqe_gain":        get_stat("niqe_gain")[0],
        
        # 4. Hiệu năng phần cứng
        "avg_latency_ms":       avg_lat,
        "throughput_fps":       fps_val,
        "elapsed_sec_total":    round(total_time_sec, 3),
        "wall_time_min":        round(total_time_sec / 60.0, 2)
    }
    
    print('═' * 80)
    print('       BÁO CÁO TỔNG QUAN CHỈ SỐ BENCHMARK (SWIFT-SRGAN Q7 — 7 CHỈ SỐ)       ')
    print('═' * 80)
    for k, v in summary.items():
        print(f'  {k:<26s}: {v}')
    print('═' * 80)
    
    df_ok = pd.DataFrame(ok_results)
    if 'dataset' in df_ok.columns:
        print('\n📊 SO SÁNH GIỮA CÁC TẬP DỮ LIỆU:')
        for ds_name, grp in df_ok.groupby('dataset'):
            lp_s = f"LPIPS={grp['lpips'].mean():.3f}" if 'lpips' in grp.columns else ""
            nq_s = f" | NIQE={grp['niqe_fpga'].mean():.2f}" if 'niqe_fpga' in grp.columns and grp['niqe_fpga'].notna().any() else ""
            print(f'  ► [{ds_name}] ({len(grp)} ảnh): PSNR={grp["psnr_fpga_db"].mean():.2f}dB (Gain: {grp["psnr_gain_db"].mean():+.2f}dB) | {lp_s}{nq_s}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — 💾 Ghi Đè File JSON & CSV Chuẩn (Swift-SRGAN)      ║
# ╚══════════════════════════════════════════════════════════════╝

OUTPUT_JSON = '/kaggle/working/swift_srgan_benchmark_q7.json'
OUTPUT_CSV  = '/kaggle/working/swift_srgan_benchmark_q7.csv'
CHECKPOINT_JSON = '/kaggle/working/benchmark_checkpoint.json'

final_payload = {
    "elapsed_sec_accumulated": summary.get("elapsed_sec_total", 0.0),
    "summary": summary,
    "per_image_results": results
}

# 1. Ghi ra file JSON chuẩn hóa
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(final_payload, f, indent=2, ensure_ascii=False)

json_size_mb = os.path.getsize(OUTPUT_JSON) / (1024 * 1024)
print('═' * 70)
print(f'✓ File JSON đã lưu : {OUTPUT_JSON} ({json_size_mb:.2f} MB, {len(results)} ảnh)')
if summary.get("avg_lpips") is not None:
    print(f'  ► avg_lpips trong summary : {summary["avg_lpips"]}')
if summary.get("avg_niqe_srgan") is not None:
    print(f'  ► avg_niqe trong summary  : {summary["avg_niqe_srgan"]}')

# 2. Ghi ra file CSV (kèm đầy đủ cột PSNR, MSE, RMSE, SSIM, MS-SSIM, LPIPS, NIQE, EPI, Mean, STD)
if ok_results:
    df_out = pd.DataFrame(ok_results)
    df_out.to_csv(OUTPUT_CSV, index=False)
    print(f'✓ File CSV đã lưu  : {OUTPUT_CSV} (Đầy đủ mọi cột mở bằng Excel)')

# 3. Xóa checkpoint tạm
if os.path.exists(CHECKPOINT_JSON):
    os.remove(CHECKPOINT_JSON)
    print('✓ Đã dọn dẹp checkpoint tạm.')
print('═' * 70)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — 🖼️ Trực Quan Hóa & Đóng Gói File Tải Về           ║
# ╚══════════════════════════════════════════════════════════════╝

if ok_results:
    df = pd.DataFrame(ok_results)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=150)
    
    # 1. Histogram PSNR Gain
    axes[0, 0].hist(df['psnr_gain_db'], bins=40, color='#2ca02c', edgecolor='black', alpha=0.7)
    axes[0, 0].axvline(0, color='red', linestyle='--', label='0 dB Baseline')
    axes[0, 0].set_title('Phân Bố PSNR Gain (dB) so với Bicubic', fontweight='bold')
    axes[0, 0].set_xlabel('PSNR Gain (dB)')
    axes[0, 0].legend()
    
    # 2. Histogram SSIM
    axes[0, 1].hist(df['ssim_fpga'], bins=40, color='#1f77b4', edgecolor='black', alpha=0.7)
    axes[0, 1].set_title('Phân Bố SSIM của Swift-SRGAN', fontweight='bold')
    axes[0, 1].set_xlabel('SSIM Index')
    
    # 3. Histogram LPIPS
    if 'lpips' in df.columns and df['lpips'].notna().any():
        axes[1, 0].hist(df['lpips'].dropna(), bins=40, color='#ff7f0e', edgecolor='black', alpha=0.7)
        axes[1, 0].set_title('Phân Bố LPIPS (AlexNet) — Càng thấp càng tốt', fontweight='bold')
        axes[1, 0].set_xlabel('LPIPS Score')
        
    # 4. Histogram NIQE
    if 'niqe_fpga' in df.columns and df['niqe_fpga'].notna().any():
        axes[1, 1].hist(df['niqe_fpga'].dropna(), bins=40, color='#9467bd', edgecolor='black', alpha=0.7)
        axes[1, 1].set_title('Phân Bố NIQE (No-Reference) — Càng thấp càng tốt', fontweight='bold')
        axes[1, 1].set_xlabel('NIQE Score')
        
    plt.tight_layout()
    plot_path = '/kaggle/working/srgan_benchmark_distribution.png'
    plt.savefig(plot_path)
    plt.show()
    print(f'✓ Đã lưu biểu đồ phân bố tại: {plot_path}')
    
    # Đóng gói ảnh PNG nếu có
    if SAVE_PNG_IMAGES and os.path.exists(OUTPUT_DIR):
        zip_path = '/kaggle/working/srgan_output_images.zip'
        print('[INFO] Đang nén thư mục ảnh đầu ra...')
        shutil.make_archive('/kaggle/working/srgan_output_images', 'zip', OUTPUT_DIR)
        print(f'✓ Đã tạo file ZIP ảnh: {zip_path} ({os.path.getsize(zip_path)/(1024*1024):.1f} MB)')